# Experimento de Reglas de Asociacion

Dataset: **Social Media User Activity Dataset**

Objetivo: limpiar y preparar el dataset como matriz binaria, ejecutar **Apriori**, **FP-Growth** y **Eclat**, comparar la calidad de sus reglas y elegir el mejor algoritmo sin usar runtime como criterio.

## 1. Configuracion sin warnings visibles

In [1]:
import os
import warnings
import logging

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

os.environ["PYTHONWARNINGS"] = "ignore"
warnings.simplefilter("ignore")
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
logging.getLogger("py.warnings").setLevel(logging.ERROR)

## 2. Instalacion de dependencias

La salida se captura para evitar logs extensos en el notebook.

In [2]:
%%capture
if IN_COLAB:
    get_ipython().system("pip install -q kaggle gdown mlxtend pyECLAT")

## 3. Cargar modulo del proyecto

Si abriste el notebook desde GitHub, Colab puede cargar solo el `.ipynb`. Esta celda clona el repositorio si falta la carpeta `src/`.

In [3]:
from pathlib import Path
import sys
import zipfile

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 160)

REPO_URL = "https://github.com/Shtolaa/ing_Datos_Experimento.git"
REPO_BRANCH = "dev"
REPO_DIR = Path("/content/ing_Datos_Experimento")
CWD = Path.cwd()
PROJECT_ROOT = CWD if (CWD / "src").exists() else CWD.parent if (CWD.parent / "src").exists() else None
LOCAL_SRC = PROJECT_ROOT / "src" if PROJECT_ROOT is not None else Path("../src")
COLAB_SRC = REPO_DIR / "src"

if IN_COLAB:
    if not LOCAL_SRC.exists() and not COLAB_SRC.exists():
        get_ipython().system(f"git clone -q -b {REPO_BRANCH} {REPO_URL} {REPO_DIR}")
    elif COLAB_SRC.exists():
        get_ipython().system(f"git -C {REPO_DIR} pull -q origin {REPO_BRANCH}")

PROJECT_SRC = LOCAL_SRC if LOCAL_SRC.exists() else COLAB_SRC
if not PROJECT_SRC.exists():
    raise FileNotFoundError(f"No se encontro la carpeta src/: {PROJECT_SRC}")
if PROJECT_ROOT is None and PROJECT_SRC == COLAB_SRC:
    PROJECT_ROOT = REPO_DIR

sys.path.insert(0, str(PROJECT_SRC.resolve()))
print(f"Modulo cargado desde: {PROJECT_SRC.resolve()}")

Modulo cargado desde: C:\Users\00rap\Documents\GitHub\ing_Datos_Experimento\src


In [4]:
from social_media_activity_pipeline import (
    TARGET_COLUMN,
    analyze_column_cardinality,
    choose_best_algorithm,
    clean_selected_data,
    create_binary_matrix,
    default_analysis_columns,
    discretize_numeric_columns,
    drop_columns,
    filter_happiness_rules,
    load_dataset,
    normalize_categorical_values,
    normalize_column_names,
    prune_binary_matrix_by_support,
    reduce_rare_categories,
    report_duplicates,
    report_missing_values,
    rules_to_readable,
    run_algorithm_experiment,
    sample_dataframe,
    summarize_algorithm_result,
    split_column_types,
    suggest_columns_to_drop,
    summarize_columns,
    validate_binary_matrix,
)

## 4. Parametros del experimento

Primero se prueba con una muestra. Para evitar agotar RAM, la corrida completa debe hacerse de forma gradual y preferiblemente sin Eclat.

In [5]:
TARGET_COLUMN = "self_reported_happiness"
USE_SAMPLE = True
SAMPLE_SIZE = 50_000 if IN_COLAB else 100_000
RANDOM_STATE = 42
MAX_BINS = 4 if IN_COLAB else 5
MIN_SUPPORT = 0.03 if IN_COLAB else 0.02
MIN_CONFIDENCE = 0.40
MIN_LIFT = 1.00
MAX_ITEMSET_LENGTH = 3
ECLAT_MAX_COMBINATION = 2
ECLAT_USE_SEPARATE_SAMPLE = True
ECLAT_SAMPLE_SIZE = 10_000 if IN_COLAB else 20_000
ALGORITHMS_TO_RUN = ["fp_growth", "apriori", "eclat"]
USE_SPARSE_BINARY_MATRIX = True
FILTER_ITEMS_BY_SUPPORT = True
GROUP_RARE_CATEGORIES = True
MIN_CATEGORY_FREQUENCY = 0.005

DATASET_SLUG = "sadiajavedd/social-media-user-activity-dataset"
GOOGLE_DRIVE_FILE_ID = "1QYfTGOepKVSNxmSRV2xw6rcGe9L7Ujpr"
DATA_DIR = Path("/content/data") if IN_COLAB else PROJECT_ROOT / "data"
OUTPUT_DIR = Path("/content/outputs") if IN_COLAB else PROJECT_ROOT / "outputs"
LOCAL_CSV_PATH = PROJECT_ROOT / "instagram_usage_lifestyle.csv"
LOCAL_ZIP_PATH = PROJECT_ROOT / "archive.zip"
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 5. Descargar o cargar dataset

En local se usa primero `../instagram_usage_lifestyle.csv` o `../archive.zip`. En Colab se intenta Google Drive y luego Kaggle como respaldo.

In [6]:
def first_existing_path(paths):
    for path in paths:
        if path.exists():
            return path
    return None

CSV_PATH = first_existing_path([
    LOCAL_CSV_PATH,
    Path("instagram_usage_lifestyle.csv"),
    DATA_DIR / "instagram_usage_lifestyle.csv",
])

if CSV_PATH is None:
    zip_candidates = [LOCAL_ZIP_PATH, Path("archive.zip")]

    if IN_COLAB:
        drive_zip_path = DATA_DIR / "archive.zip"
        if not drive_zip_path.exists():
            print("Descargando ZIP desde Google Drive...")
            drive_url = f"https://drive.google.com/uc?id={GOOGLE_DRIVE_FILE_ID}"
            get_ipython().system(f"gdown '{drive_url}' -O '{drive_zip_path}' --quiet")
        zip_candidates.append(drive_zip_path)

    for zip_path in zip_candidates:
        if zip_path.exists():
            print(f"Extrayendo {zip_path}...")
            with zipfile.ZipFile(zip_path, "r") as zip_ref:
                zip_ref.extractall(DATA_DIR)

    csv_files = sorted(DATA_DIR.glob("*.csv"))
    CSV_PATH = csv_files[0] if csv_files else None

if CSV_PATH is None and IN_COLAB:
    from google.colab import files

    if not Path("/root/.kaggle/kaggle.json").exists():
        print("No se encontro CSV/ZIP. Sube kaggle.json para descargar desde Kaggle.")
        uploaded = files.upload()
        if "kaggle.json" in uploaded:
            Path("/root/.kaggle").mkdir(parents=True, exist_ok=True)
            Path("/root/.kaggle/kaggle.json").write_bytes(uploaded["kaggle.json"])
            get_ipython().system("chmod 600 /root/.kaggle/kaggle.json")

    get_ipython().system(f"kaggle datasets download -d {DATASET_SLUG} -p {DATA_DIR} --force")
    for zip_path in DATA_DIR.glob("*.zip"):
        with zipfile.ZipFile(zip_path, "r") as zip_ref:
            zip_ref.extractall(DATA_DIR)

    csv_files = sorted(DATA_DIR.glob("*.csv"))
    CSV_PATH = csv_files[0] if csv_files else None

if CSV_PATH is None:
    raise FileNotFoundError("No se encontro el CSV. Coloca instagram_usage_lifestyle.csv o archive.zip en la raiz del proyecto.")

CSV_PATH = CSV_PATH.resolve()
print(f"CSV usado: {CSV_PATH}")

CSV usado: C:\Users\00rap\Documents\GitHub\ing_Datos_Experimento\instagram_usage_lifestyle.csv


In [7]:
CSV_PATH

WindowsPath('C:/Users/00rap/Documents/GitHub/ing_Datos_Experimento/instagram_usage_lifestyle.csv')

In [8]:
DATA_DIR

WindowsPath('C:/Users/00rap/Documents/GitHub/ing_Datos_Experimento/data')

## 6. Carga, normalizacion y EDA inicial

In [9]:
df_raw = load_dataset(CSV_PATH)
df = normalize_column_names(df_raw)

print(f"Dataset original: {df_raw.shape}")
print(f"Dataset con columnas normalizadas: {df.shape}")
df.head()

Dataset original: (1547896, 58)
Dataset con columnas normalizadas: (1547896, 58)


,user_id,app_name,age,gender,country,urban_rural,income_level,employment_status,education_level,relationship_status,has_children,exercise_hours_per_week,sleep_hours_per_night,diet_quality,smoking,alcohol_frequency,perceived_stress_score,self_reported_happiness,body_mass_index,blood_pressure_systolic,blood_pressure_diastolic,daily_steps_count,weekly_work_hours,hobbies_count,social_events_per_month,books_read_per_year,volunteer_hours_per_month,travel_frequency_per_year,daily_active_minutes_instagram,sessions_per_day,posts_created_per_week,reels_watched_per_day,stories_viewed_per_day,likes_given_per_day,comments_written_per_day,dms_sent_per_week,dms_received_per_week,ads_viewed_per_day,ads_clicked_per_day,time_on_feed_per_day,time_on_explore_per_day,time_on_messages_per_day,time_on_reels_per_day,followers_count,following_count,uses_premium_features,notification_response_rate,account_creation_year,last_login_date,average_session_length_minutes,content_type_preference,preferred_content_theme,privacy_setting_level,two_factor_auth_enabled,biometric_login_used,linked_accounts_count,subscription_status,user_engagement_score
0,1,Instagram,51,Female,India,Rural,High,Retired,Bachelor’s,Single,No,7.2,7.7,Good,No,Rarely,3,8,20.8,148,86,8107,49.9,3,4,7,4.3,0,5.0,1,3,42,28,28,5,12,12,4,1,2,1,1,2,374,647,No,0.34,2015,2025-11-02,5.0,Mixed,Tech,Private,Yes,No,0,Free,7.83
1,2,Instagram,64,Female,United Kingdom,Urban,Middle,Full-time employed,Other,Divorced,No,10.9,8.6,Very poor,No,Rarely,1,1,23.5,133,84,8059,15.6,0,5,10,4.7,2,74.0,5,3,78,54,68,15,18,10,11,1,31,19,16,19,2585,3511,No,0.56,2018,2025-03-22,14.8,Photos,Fashion,Public,No,No,3,Free,1.43
2,3,Instagram,41,Female,Canada,Urban,Middle,Student,Bachelor’s,In a relationship,No,5.0,6.7,Good,No,Rarely,4,10,28.6,135,88,7872,31.8,4,5,14,1.5,2,5.0,1,7,29,26,25,6,12,13,4,0,3,1,1,1,3414,6761,No,0.73,2011,2025-08-10,5.0,Mixed,Other,Public,Yes,Yes,1,Free,9.67
3,4,Instagram,27,Non-binary,South Korea,Urban,Middle,Unemployed,Master’s,In a relationship,No,10.6,6.5,Poor,Yes,Never,18,1,22.5,105,73,7801,43.4,2,3,13,3.3,4,233.0,9,5,241,109,132,36,31,32,33,3,108,64,52,64,617,1193,No,0.73,2019,2025-03-31,25.9,Stories,Tech,Private,No,No,1,Free,0.94
4,5,Instagram,55,Male,India,Urban,Upper-middle,Full-time employed,Bachelor’s,Single,No,7.7,6.8,Average,No,Never,19,1,28.1,146,90,8005,50.2,2,2,12,4.5,3,184.0,14,5,146,113,103,36,29,37,20,5,78,55,22,55,1157,1072,Yes,0.65,2017,2025-03-19,13.1,Videos,Food,Public,Yes,No,0,Free,1.03


In [10]:
column_analysis = analyze_column_cardinality(df)
column_analysis

,column,dtype,missing_count,missing_pct,unique_count,unique_ratio,example_values,suggested_action,reason
0,account_creation_year,int64,0,0.0,16,0.000010,"[2015, 2018, 2011, 2019, 2017]",keep,usable for association rules
1,ads_clicked_per_day,int64,0,0.0,21,0.000014,"[1, 0, 3, 5, 9]",keep,usable for association rules
2,ads_viewed_per_day,int64,0,0.0,51,0.000033,"[4, 11, 33, 20, 16]",keep,usable for association rules
3,age,int64,0,0.0,53,0.000034,"[51, 64, 41, 27, 55]",keep,usable for association rules
4,alcohol_frequency,str,0,0.0,5,0.000003,"[Rarely, Never, Several times a week, Weekly, Daily]",keep,usable for association rules
5,app_name,str,0,0.0,1,0.000001,[Instagram],drop_candidate,constant or nearly empty column
6,average_session_length_minutes,float64,0,0.0,342,0.000221,"[5.0, 14.8, 25.9, 13.1, 16.3]",keep,usable for association rules
7,biometric_login_used,str,0,0.0,2,0.000001,"[No, Yes]",keep,usable for association rules
8,blood_pressure_diastolic,int64,0,0.0,40,0.000026,"[86, 84, 88, 73, 90]",keep,usable for association rules
9,blood_pressure_systolic,int64,0,0.0,70,0.000045,"[148, 133, 135, 105, 146]",keep,usable for association rules


In [11]:
display(report_missing_values(df).head(20))
report_duplicates(df)

,missing_count,missing_pct
user_id,0,0.0
app_name,0,0.0
age,0,0.0
gender,0,0.0
country,0,0.0
urban_rural,0,0.0
income_level,0,0.0
employment_status,0,0.0
education_level,0,0.0
relationship_status,0,0.0


{'duplicate_count': 0, 'duplicate_pct': 0.0}

## 7. Eliminacion de columnas que no aportan

La lista es editable. Se sugiere eliminar IDs, columnas constantes, cardinalidad extrema o demasiados nulos.

In [12]:
suggested_drop_columns = suggest_columns_to_drop(df, target_column=TARGET_COLUMN)
suggested_drop_columns

['user_id', 'app_name', 'last_login_date']

In [13]:
# Edita esta lista si quieres conservar o eliminar columnas adicionales.
columns_to_drop = suggested_drop_columns.copy()

df_reduced = drop_columns(df, columns_to_drop)
print(f"Columnas antes: {df.shape[1]}")
print(f"Columnas despues: {df_reduced.shape[1]}")
summarize_columns(df_reduced)

Columnas antes: 58
Columnas despues: 55


,column,dtype,missing_count,missing_pct,unique_count,unique_ratio,example_values
0,account_creation_year,int64,0,0.0,16,0.000010,"[2015, 2018, 2011, 2019, 2017]"
1,ads_clicked_per_day,int64,0,0.0,21,0.000014,"[1, 0, 3, 5, 9]"
2,ads_viewed_per_day,int64,0,0.0,51,0.000033,"[4, 11, 33, 20, 16]"
3,age,int64,0,0.0,53,0.000034,"[51, 64, 41, 27, 55]"
4,alcohol_frequency,str,0,0.0,5,0.000003,"[Rarely, Never, Several times a week, Weekly, Daily]"
5,average_session_length_minutes,float64,0,0.0,342,0.000221,"[5.0, 14.8, 25.9, 13.1, 16.3]"
6,biometric_login_used,str,0,0.0,2,0.000001,"[No, Yes]"
7,blood_pressure_diastolic,int64,0,0.0,40,0.000026,"[86, 84, 88, 73, 90]"
8,blood_pressure_systolic,int64,0,0.0,70,0.000045,"[148, 133, 135, 105, 146]"
9,body_mass_index,float64,0,0.0,286,0.000185,"[20.8, 23.5, 28.6, 22.5, 28.1]"


## 8. Seleccion final de columnas

Se propone una lista basada en la documentacion del proyecto, pero puedes editarla.

In [14]:
suggested_columns = default_analysis_columns(df_reduced)
if TARGET_COLUMN in df_reduced.columns and TARGET_COLUMN not in suggested_columns:
    suggested_columns.append(TARGET_COLUMN)

selected_columns = suggested_columns if suggested_columns else df_reduced.columns.tolist()
df_selected = df_reduced[selected_columns].copy()
selected_columns

['age',
 'gender',
 'country',
 'urban_rural',
 'income_level',
 'employment_status',
 'education_level',
 'relationship_status',
 'has_children',
 'exercise_hours_per_week',
 'sleep_hours_per_night',
 'diet_quality',
 'smoking',
 'perceived_stress_score',
 'self_reported_happiness',
 'body_mass_index',
 'daily_steps_count',
 'weekly_work_hours',
 'sessions_per_day',
 'average_session_length_minutes',
 'posts_created_per_week',
 'likes_given_per_day',
 'comments_written_per_day',
 'dms_sent_per_week',
 'followers_count',
 'following_count',
 'time_on_feed_per_day',
 'time_on_explore_per_day',
 'time_on_messages_per_day',
 'time_on_reels_per_day',
 'content_type_preference',
 'preferred_content_theme',
 'privacy_setting_level']

## 9. Limpieza, normalizacion de categorias y muestra de prueba

In [15]:
df_clean = clean_selected_data(df_selected, numeric_strategy="median", categorical_strategy="unknown")
df_clean = normalize_categorical_values(df_clean)
df_work = sample_dataframe(df_clean, use_sample=USE_SAMPLE, sample_size=SAMPLE_SIZE, random_state=RANDOM_STATE)

print(f"Dataset limpio: {df_clean.shape}")
print(f"Dataset usado en esta prueba: {df_work.shape}")
df_work.head()

Dataset limpio: (1547896, 33)
Dataset usado en esta prueba: (100000, 33)


,age,gender,country,urban_rural,income_level,employment_status,education_level,relationship_status,has_children,exercise_hours_per_week,sleep_hours_per_night,diet_quality,smoking,perceived_stress_score,self_reported_happiness,body_mass_index,daily_steps_count,weekly_work_hours,sessions_per_day,average_session_length_minutes,posts_created_per_week,likes_given_per_day,comments_written_per_day,dms_sent_per_week,followers_count,following_count,time_on_feed_per_day,time_on_explore_per_day,time_on_messages_per_day,time_on_reels_per_day,content_type_preference,preferred_content_theme,privacy_setting_level
0,43,male,brazil,rural,low,student,some_college,single,no,1.3,6.2,average,former,31,10,31.1,7938,26.6,9,22.3,6,133,37,22,292,478,105,25,47,56,mixed,fashion,public
1,17,male,germany,urban,low,student,some_college,single,yes,18.1,8.0,average,no,25,10,18.2,7995,28.3,7,31.7,14,139,39,37,1359,1232,100,61,32,70,stories,travel,public
2,52,female,united_kingdom,urban,middle,full_time_employed,high_school,married,no,4.6,7.9,average,no,36,10,27.9,7817,46.1,11,17.9,1,143,38,29,214,132,96,53,46,68,reels,fashion,public
3,24,male,canada,suburban,middle,part_time,other,in_a_relationship,no,7.7,6.1,average,no,2,10,22.6,8042,52.5,1,11.0,3,26,8,12,851,594,5,2,1,4,reels,tech,public
4,31,male,south_korea,rural,high,freelancer,bachelors,single,no,8.9,7.0,average,no,17,9,21.0,8047,31.7,21,11.8,6,162,33,37,828,418,137,27,55,50,mixed,art,private


In [16]:
if GROUP_RARE_CATEGORIES:
    categorical_columns = split_column_types(df_work)["categorical"]
    df_work = reduce_rare_categories(
        df_work,
        categorical_columns=categorical_columns,
        min_frequency=MIN_CATEGORY_FREQUENCY,
    )

summarize_columns(df_work)

,column,dtype,missing_count,missing_pct,unique_count,unique_ratio,example_values
0,age,int64,0,0.0,53,0.00053,"[43, 17, 52, 24, 31]"
1,average_session_length_minutes,float64,0,0.0,341,0.00341,"[22.3, 31.7, 17.9, 11.0, 11.8]"
2,body_mass_index,float64,0,0.0,256,0.00256,"[31.1, 18.2, 27.9, 22.6, 21.0]"
3,comments_written_per_day,int64,0,0.0,81,0.00081,"[37, 39, 38, 8, 33]"
4,content_type_preference,str,0,0.0,6,0.00006,"[mixed, stories, reels, videos, live]"
5,country,str,0,0.0,10,0.00010,"[brazil, germany, united_kingdom, canada, south_korea]"
6,daily_steps_count,int64,0,0.0,644,0.00644,"[7938, 7995, 7817, 8042, 8047]"
7,diet_quality,str,0,0.0,5,0.00005,"[average, good, very_poor, excellent, poor]"
8,dms_sent_per_week,int64,0,0.0,80,0.00080,"[22, 37, 29, 12, 16]"
9,education_level,str,0,0.0,6,0.00006,"[some_college, high_school, other, bachelors, masters]"


## 10. Discretizacion y matriz binaria

Las variables numericas se separan por cuantiles en maximo `MAX_BINS` intervalos. La matriz binaria se crea en formato sparse y se filtran items con soporte menor a `MIN_SUPPORT`.

In [17]:
df_discretized = discretize_numeric_columns(df_work, max_bins=MAX_BINS)
binary_matrix = create_binary_matrix(df_discretized, sparse=USE_SPARSE_BINARY_MATRIX)

if FILTER_ITEMS_BY_SUPPORT:
    original_columns = binary_matrix.shape[1]
    binary_matrix = prune_binary_matrix_by_support(binary_matrix, min_support=MIN_SUPPORT)
    print(f"Items antes/despues de filtrar por soporte: {original_columns} -> {binary_matrix.shape[1]}")

validate_binary_matrix(binary_matrix)

Items antes/despues de filtrar por soporte: 166 -> 166


{'rows': 100000,
 'columns': 166,
 'dtype_counts': {'Sparse[bool, False]': 166},
 'has_missing_values': False}

## 11. Ejecutar Apriori, FP-Growth y Eclat

La comparacion no usa runtime. Para controlar RAM, los algoritmos se ejecutan uno por uno, se guardan sus resultados y Eclat usa una muestra separada.

In [18]:
import gc

summary_rows = []
happiness_rules_for_display = {}

for algorithm in ALGORITHMS_TO_RUN:
    print(f"\nEjecutando {algorithm}...")
    algorithm_binary_matrix = binary_matrix

    if algorithm == "eclat" and ECLAT_USE_SEPARATE_SAMPLE:
        eclat_rows = min(ECLAT_SAMPLE_SIZE, len(df_discretized))
        print(f"Eclat se ejecuta con muestra separada de {eclat_rows:,} filas para controlar RAM.")
        eclat_df = sample_dataframe(
            df_discretized,
            use_sample=True,
            sample_size=eclat_rows,
            random_state=RANDOM_STATE,
        )
        algorithm_binary_matrix = create_binary_matrix(eclat_df, sparse=False)
        if FILTER_ITEMS_BY_SUPPORT:
            original_columns = algorithm_binary_matrix.shape[1]
            algorithm_binary_matrix = prune_binary_matrix_by_support(algorithm_binary_matrix, min_support=MIN_SUPPORT)
            print(f"Items Eclat antes/despues de filtrar: {original_columns} -> {algorithm_binary_matrix.shape[1]}")

    if algorithm_binary_matrix.shape[1] == 0:
        print(f"Se omite {algorithm}: no quedaron items con soporte >= {MIN_SUPPORT}.")
        if algorithm_binary_matrix is not binary_matrix:
            del algorithm_binary_matrix
        if "eclat_df" in locals():
            del eclat_df
        gc.collect()
        continue

    result = run_algorithm_experiment(
        binary_df=algorithm_binary_matrix,
        algorithm=algorithm,
        min_support=MIN_SUPPORT,
        min_confidence=MIN_CONFIDENCE,
        min_lift=MIN_LIFT,
        target_column=TARGET_COLUMN,
        eclat_max_combination=ECLAT_MAX_COMBINATION,
        max_itemset_length=MAX_ITEMSET_LENGTH,
    )

    algorithm_key = result["algorithm"]
    result["itemsets"].to_csv(OUTPUT_DIR / f"{algorithm_key}_itemsets.csv", index=False)
    rules_to_readable(result["rules"]).to_csv(OUTPUT_DIR / f"{algorithm_key}_rules.csv", index=False)
    readable_happiness_rules = rules_to_readable(result["happiness_rules"])
    readable_happiness_rules.to_csv(OUTPUT_DIR / f"{algorithm_key}_happiness_rules.csv", index=False)

    summary_rows.append(
        summarize_algorithm_result(
            algorithm_key,
            result["itemsets"],
            result["rules"],
            result["happiness_rules"],
        )
    )
    happiness_rules_for_display[algorithm_key] = readable_happiness_rules.head(15)

    del result, readable_happiness_rules
    if algorithm_binary_matrix is not binary_matrix:
        del algorithm_binary_matrix
    if "eclat_df" in locals():
        del eclat_df
    gc.collect()

if not summary_rows:
    raise ValueError("Ningun algoritmo genero resultados. Reduce MIN_SUPPORT o revisa la matriz binaria.")

comparison = pd.DataFrame(summary_rows).sort_values(
    ["happiness_rules_count", "avg_lift", "avg_confidence", "rules_count"],
    ascending=[False, False, False, False],
).reset_index(drop=True)
comparison.to_csv(OUTPUT_DIR / "algorithm_comparison.csv", index=False)
comparison


Ejecutando fp_growth...

Ejecutando apriori...

Ejecutando eclat...
Eclat se ejecuta con muestra separada de 20,000 filas para controlar RAM.
Items Eclat antes/despues de filtrar: 166 -> 165


,algorithm,frequent_itemsets_count,rules_count,happiness_rules_count,avg_support,avg_confidence,avg_lift,max_lift
0,fp_growth,57676,41550,2427,0.034387,0.587433,1.768010,4.638958
1,apriori,57676,41550,2427,0.034387,0.587433,1.768010,4.638958
2,eclat,10371,849,26,0.098937,0.511128,1.234933,1.696378


In [19]:
best_algorithm = choose_best_algorithm(comparison)
print(f"Mejor algoritmo segun calidad de reglas: {best_algorithm}")

Mejor algoritmo segun calidad de reglas: fp_growth


## 12. Reglas asociadas a felicidad

In [20]:
columns_to_show = ["antecedents", "consequents", "support", "confidence", "lift"]

for algorithm, readable_rules in happiness_rules_for_display.items():
    print(f"\n=== {algorithm.upper()} ===")
    if readable_rules.empty:
        print("Sin reglas relacionadas con felicidad para mostrar.")
    else:
        display(readable_rules[columns_to_show])


=== FP_GROWTH ===


,antecedents,consequents,support,confidence,lift
0,perceived_stress_score=perceived_stress_score__-0.001_to_8 AND self_reported_happiness=self_reported_happiness__8_to_10,time_on_feed_per_day=time_on_feed_per_day__2_to_41,0.04127,0.947646,4.638958
1,perceived_stress_score=perceived_stress_score__32_to_40 AND self_reported_happiness=self_reported_happiness__0.999_to_3,likes_given_per_day=likes_given_per_day__171_to_317,0.05325,0.908548,4.628365
2,perceived_stress_score=perceived_stress_score__-0.001_to_8 AND self_reported_happiness=self_reported_happiness__8_to_10,time_on_messages_per_day=time_on_messages_per_day__0.999_to_13,0.04042,0.928129,4.608384
3,perceived_stress_score=perceived_stress_score__-0.001_to_8 AND self_reported_happiness=self_reported_happiness__8_to_10,likes_given_per_day=likes_given_per_day__11_to_66,0.04102,0.941906,4.601621
4,likes_given_per_day=likes_given_per_day__11_to_66 AND self_reported_happiness=self_reported_happiness__8_to_10,time_on_feed_per_day=time_on_feed_per_day__2_to_41,0.06878,0.938208,4.592753
5,self_reported_happiness=self_reported_happiness__8_to_10 AND time_on_feed_per_day=time_on_feed_per_day__2_to_41,likes_given_per_day=likes_given_per_day__11_to_66,0.06878,0.936674,4.576063
6,self_reported_happiness=self_reported_happiness__8_to_10 AND time_on_reels_per_day=time_on_reels_per_day__0.999_to_24,time_on_feed_per_day=time_on_feed_per_day__2_to_41,0.06877,0.934756,4.575857
7,perceived_stress_score=perceived_stress_score__-0.001_to_8 AND self_reported_happiness=self_reported_happiness__8_to_10,time_on_reels_per_day=time_on_reels_per_day__0.999_to_24,0.04108,0.943284,4.564423
8,self_reported_happiness=self_reported_happiness__8_to_10 AND time_on_feed_per_day=time_on_feed_per_day__2_to_41,time_on_reels_per_day=time_on_reels_per_day__0.999_to_24,0.06877,0.936538,4.531783
9,likes_given_per_day=likes_given_per_day__11_to_66 AND self_reported_happiness=self_reported_happiness__6_to_8,time_on_feed_per_day=time_on_feed_per_day__2_to_41,0.05161,0.924579,4.526038



=== APRIORI ===


,antecedents,consequents,support,confidence,lift
0,perceived_stress_score=perceived_stress_score__-0.001_to_8 AND self_reported_happiness=self_reported_happiness__8_to_10,time_on_feed_per_day=time_on_feed_per_day__2_to_41,0.04127,0.947646,4.638958
1,perceived_stress_score=perceived_stress_score__32_to_40 AND self_reported_happiness=self_reported_happiness__0.999_to_3,likes_given_per_day=likes_given_per_day__171_to_317,0.05325,0.908548,4.628365
2,perceived_stress_score=perceived_stress_score__-0.001_to_8 AND self_reported_happiness=self_reported_happiness__8_to_10,time_on_messages_per_day=time_on_messages_per_day__0.999_to_13,0.04042,0.928129,4.608384
3,perceived_stress_score=perceived_stress_score__-0.001_to_8 AND self_reported_happiness=self_reported_happiness__8_to_10,likes_given_per_day=likes_given_per_day__11_to_66,0.04102,0.941906,4.601621
4,likes_given_per_day=likes_given_per_day__11_to_66 AND self_reported_happiness=self_reported_happiness__8_to_10,time_on_feed_per_day=time_on_feed_per_day__2_to_41,0.06878,0.938208,4.592753
5,self_reported_happiness=self_reported_happiness__8_to_10 AND time_on_feed_per_day=time_on_feed_per_day__2_to_41,likes_given_per_day=likes_given_per_day__11_to_66,0.06878,0.936674,4.576063
6,self_reported_happiness=self_reported_happiness__8_to_10 AND time_on_reels_per_day=time_on_reels_per_day__0.999_to_24,time_on_feed_per_day=time_on_feed_per_day__2_to_41,0.06877,0.934756,4.575857
7,perceived_stress_score=perceived_stress_score__-0.001_to_8 AND self_reported_happiness=self_reported_happiness__8_to_10,time_on_reels_per_day=time_on_reels_per_day__0.999_to_24,0.04108,0.943284,4.564423
8,self_reported_happiness=self_reported_happiness__8_to_10 AND time_on_feed_per_day=time_on_feed_per_day__2_to_41,time_on_reels_per_day=time_on_reels_per_day__0.999_to_24,0.06877,0.936538,4.531783
9,likes_given_per_day=likes_given_per_day__11_to_66 AND self_reported_happiness=self_reported_happiness__6_to_8,time_on_feed_per_day=time_on_feed_per_day__2_to_41,0.05161,0.924579,4.526038



=== ECLAT ===


,antecedents,consequents,support,confidence,lift
0,comments_written_per_day=comments_written_per_day__50_to_80,self_reported_happiness=self_reported_happiness__0.999_to_3,0.09515,0.501449,1.696378
1,likes_given_per_day=likes_given_per_day__171_to_317,self_reported_happiness=self_reported_happiness__0.999_to_3,0.09770,0.501026,1.694945
2,time_on_feed_per_day=time_on_feed_per_day__145_to_321,self_reported_happiness=self_reported_happiness__0.999_to_3,0.09800,0.499109,1.688460
3,self_reported_happiness=self_reported_happiness__8_to_10,sessions_per_day=sessions_per_day__0.999_to_4,0.08080,0.409632,1.671628
4,dms_sent_per_week=dms_sent_per_week__40_to_80,self_reported_happiness=self_reported_happiness__0.999_to_3,0.08820,0.486486,1.645759
5,time_on_reels_per_day=time_on_reels_per_day__87_to_195,self_reported_happiness=self_reported_happiness__0.999_to_3,0.09270,0.481934,1.630359
6,time_on_messages_per_day=time_on_messages_per_day__51_to_128,self_reported_happiness=self_reported_happiness__0.999_to_3,0.09295,0.473751,1.602677
7,time_on_explore_per_day=time_on_explore_per_day__59_to_151,self_reported_happiness=self_reported_happiness__0.999_to_3,0.08705,0.459974,1.556068
8,sessions_per_day=sessions_per_day__16_to_50,self_reported_happiness=self_reported_happiness__0.999_to_3,0.07510,0.445169,1.505984
9,self_reported_happiness=self_reported_happiness__8_to_10,posts_created_per_week=posts_created_per_week__-0.001_to_3,0.07900,0.400507,1.246326


## 13. Exportar resultados

In [21]:
print(f"Resultados guardados en: {OUTPUT_DIR.resolve()}")
sorted(OUTPUT_DIR.glob("*.csv"))

Resultados guardados en: C:\Users\00rap\Documents\GitHub\ing_Datos_Experimento\outputs


[WindowsPath('C:/Users/00rap/Documents/GitHub/ing_Datos_Experimento/outputs/algorithm_comparison.csv'),
 WindowsPath('C:/Users/00rap/Documents/GitHub/ing_Datos_Experimento/outputs/apriori_happiness_rules.csv'),
 WindowsPath('C:/Users/00rap/Documents/GitHub/ing_Datos_Experimento/outputs/apriori_itemsets.csv'),
 WindowsPath('C:/Users/00rap/Documents/GitHub/ing_Datos_Experimento/outputs/apriori_rules.csv'),
 WindowsPath('C:/Users/00rap/Documents/GitHub/ing_Datos_Experimento/outputs/eclat_happiness_rules.csv'),
 WindowsPath('C:/Users/00rap/Documents/GitHub/ing_Datos_Experimento/outputs/eclat_itemsets.csv'),
 WindowsPath('C:/Users/00rap/Documents/GitHub/ing_Datos_Experimento/outputs/eclat_rules.csv'),
 WindowsPath('C:/Users/00rap/Documents/GitHub/ing_Datos_Experimento/outputs/fp_growth_happiness_rules.csv'),
 WindowsPath('C:/Users/00rap/Documents/GitHub/ing_Datos_Experimento/outputs/fp_growth_itemsets.csv'),
 WindowsPath('C:/Users/00rap/Documents/GitHub/ing_Datos_Experimento/outputs/fp_grow

## 14. Ejecucion completa opcional

Cuando la prueba con muestra funcione, aumenta el tamano gradualmente y vuelve a ejecutar desde la limpieza:

```python
SAMPLE_SIZE = 250_000
MAX_BINS = 5
MIN_SUPPORT = 0.02
ECLAT_SAMPLE_SIZE = 20_000
```

Para usar todo el dataset, prueba primero solo `ALGORITHMS_TO_RUN = ["fp_growth"]`. Eclat completo no se recomienda porque convierte la matriz a transacciones de strings y puede agotar RAM incluso fuera de Colab.